# Train the cursive transformer on `diarybank` (Colab)

Validation run: does the OCR-derived diary dataset train into legible cursive?

1. **Runtime ▸ Change runtime type ▸ T4 GPU** (free tier is fine).
2. Run the cells top to bottom. You'll be prompted for a **GitHub PAT** (repo is private) and your **W&B API key**.
3. Watch the sample images in the W&B run — legible by ~20–30k steps means the pipeline is validated.

In [ ]:
# Confirm a GPU is attached (else: Runtime ▸ Change runtime type ▸ GPU)
!nvidia-smi -L

In [ ]:
# Clone the private repo (diarybank.json.zip is committed, so it comes with the clone)
# PAT: github.com/settings/tokens -> fine-grained, read access to andreisuslov/cursivetransformer
from getpass import getpass
TOKEN = getpass('GitHub PAT (repo read): ')
!git clone -b ocr-stroke-recovery https://{TOKEN}@github.com/andreisuslov/cursivetransformer.git
%cd cursivetransformer
# scrub the token from the remote URL so it isn't left lying around
!git remote set-url origin https://github.com/andreisuslov/cursivetransformer.git
# Colab already ships torch/numpy/scipy/matplotlib -- only wandb is missing.
# (requirements.txt pins numpy>=2.1, which upgrades Colab's numpy and breaks numba; skip it.)
!pip install -q wandb

In [ ]:
# Train. WANDB_ENTITY is your W&B entity. For this account it's the TEAM slug
# (the part in parens in `wandb: Currently logged in as: suslov (...)`), not 'suslov'.
# train_size/test_size kept small: this is a "does it train at all?" validation run,
# not the full 497k-example paper run -- keeps startup to seconds on Colab's 2 vCPUs.
from getpass import getpass
WANDB_ENTITY = 'suslov-harrisburg-university-of-science-and-technology'
WANDB_KEY = getpass('W&B API key: ')
!python train.py --wandb_entity {WANDB_ENTITY} --wandb_project diary_test --wandb_api_key {WANDB_KEY} \
  --dataset_name diarybank --num_words 4 --max_seq_length 1500 \
  --train_size 40000 --test_size 1000 \
  --n_layer 5 --batch_size 32 --max_steps 50000 --learning_rate 1e-2